# Depth GUI example

Click the wells of a depth-sensitivity (RDS) target in a **popup window**,
then this notebook analyzes them. Plots stay **inside the notebook**.

**How to run**

1. Run the cells from top to bottom.
2. In *Choose your image*, either keep the example image or uncomment `IMAGE_PATH`.
3. When the well-selection window opens, click the wells, then press **Enter**.
4. The remaining cells show plots here. If saving is on, files go in a results folder.


## 1. Setup


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from qal import WellAnalyzer, WellPlotter, select_concentration_grid
from qal.data import depth_sample_1, get_target_preset
from qal.util import load_grayscale_image


## 2. Choose your image

Leave `IMAGE_PATH` as `None` to use QAL's built-in example.

To use your own file, uncomment the `IMAGE_PATH` line and paste a path to a
TIFF, BMP, PNG, or JPEG.


In [ ]:
# Uncomment the next line and put the path to your image:
# IMAGE_PATH = "/path/to/your/image.tiff"

# Leave this as None to use the built-in example image.
IMAGE_PATH = None


## 3. Saving plots and data

When `SAVE_RESULTS` is `True`:

- If you set `IMAGE_PATH`, files go next to that image in `<image_name>_results/`
- If you use the example image, files go next to this notebook in `depth_sample_1_results/`


In [ ]:
# Set this to True to write plots and CSV files.
SAVE_RESULTS = True

# File names inside the results folder (used only when SAVE_RESULTS is True):
SAVE_ROI_PLOT = "roi.png"
SAVE_DEPTH_PLOT = "depth.png"
SAVE_WELLS_CSV = "wells.csv"
SAVE_STATS_CSV = "stats.csv"


## 4. Analysis options

You can usually leave these as they are.

Click order for the first three wells (left to right along the top row):

`0.5 mm` → `1.0 mm` → `1.5 mm`


In [ ]:
# Product type: "icg", "q800", or "q700"
TARGET_TAG = "icg"

# How wells are selected. Uncomment one:
# MODE = "automatic_refinement"
MODE = "manual_live_preview"
# MODE = "manual_coordinates"

# How many wells to click (3 is enough; they must be left-to-right, top-to-bottom).
ANCHOR_COUNT = 3

# Circle size in pixels. None = estimate from well spacing.
MANUAL_ROI_DIAMETER = None

# Fraction of each well used for intensity (0.5 is recommended).
REGION_OF_WELL_TO_ANALYZE = 0.5


## 5. Load the image


In [ ]:
if IMAGE_PATH is None:
    image = depth_sample_1()
    results_dir = Path("depth_sample_1_results")
else:
    image = load_grayscale_image(IMAGE_PATH)
    image_file = Path(IMAGE_PATH).expanduser()
    results_dir = image_file.parent / f"{image_file.stem}_results"

if SAVE_RESULTS:
    results_dir.mkdir(parents=True, exist_ok=True)
    print(f"Results folder: {results_dir.resolve()}")
else:
    results_dir = None

target = get_target_preset("depth", TARGET_TAG)
well_ids = target["well_ids"]
print(f"Target: {target['display_name']}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(image, cmap="gray")
ax.set_title("Loaded image")
ax.axis("off")
plt.show()


## 6. Select wells (popup window)

This cell opens a **separate window**. Click the wells, then press **Enter** in
that window. Drag a `+` to move it.

In JupyterLab, click **Confirm** instead of Enter.


In [ ]:
# Same 3x3 well picker used for concentration targets.
wells = select_concentration_grid(
    image,
    well_ids,
    mode=MODE,
    anchor_count=ANCHOR_COUNT,
    manual_roi_diameter=MANUAL_ROI_DIAMETER,
)

# JupyterLab only: if a table was not returned, click Confirm, then uncomment:
# wells = wells.result


## 7. Put plots back in the notebook


In [ ]:
%matplotlib inline
plt.close("all")


## 8. Analyze the selected wells


In [ ]:
if not isinstance(wells, pd.DataFrame):
    raise RuntimeError(
        "No well table yet. Finish the popup (Enter or Confirm), then rerun "
        "the select-wells cell. In JupyterLab, also uncomment "
        "`wells = wells.result`."
    )

analyzer = WellAnalyzer(image, wells)
stats = analyzer.get_stats(region_of_well_to_analyze=REGION_OF_WELL_TO_ANALYZE)
stats


## 9. Show the regions of interest


In [ ]:
plotter = WellPlotter(stats, image=image)
plotter.visualize_roi()
if results_dir is not None:
    roi_path = results_dir / SAVE_ROI_PLOT
    plt.savefig(roi_path, bbox_inches="tight")
    print(f"Saved {roi_path}")


## 10. Plot depth vs intensity


In [ ]:
plotter.plot(
    graph_type="depth",
    col_to_plot="mean intensity normalized",
    fluorophore_label=target.get("fluorophore_label", "ICG-eq"),
    trendline_lib="scipy",
    save_plot=(None if results_dir is None else results_dir / SAVE_DEPTH_PLOT),
)


## 11. Save tables as CSV


In [ ]:
if results_dir is None:
    print("SAVE_RESULTS is False, so no CSV files were written.")
else:
    wells_path = results_dir / SAVE_WELLS_CSV
    stats_path = results_dir / SAVE_STATS_CSV
    wells.to_csv(wells_path, index=False)
    stats.to_csv(stats_path, index=False)
    print(f"Saved {wells_path}")
    print(f"Saved {stats_path}")
